# SuperPoint + LightGlue demo

Сравнение с классикой: те же `query` и кусок карты из `case_01_100m`, но точки/матчинг — нейросетью.

- **SuperPoint** — keypoints + дескрипторы (pretrained)
- **LightGlue** — matcher вместо BFMatcher+Lowe

Работает на **CPU** (медленно, но для демо ок). GPU подхватится сам, если есть CUDA.

## 0. Установка зависимостей

Нужны: `torch`, `kornia`, `LightGlue` (с GitHub). Ячейка ниже поставит недостающее.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

ROOT_PRE = Path.cwd().resolve()
if ROOT_PRE.name == "notebooks":
    ROOT_PRE = ROOT_PRE.parent

# Локальный клон LightGlue (если есть) — не трогаем системный OpenCV
LG = ROOT_PRE / "third_party" / "LightGlue"
if LG.exists():
    sys.path.insert(0, str(LG))


def ensure(pkg: str, pip_spec: str | None = None, user: bool = True):
    try:
        importlib.import_module(pkg)
        print(f"OK: {pkg}")
        return
    except ImportError:
        pass
    spec = pip_spec or pkg
    cmd = [sys.executable, "-m", "pip", "install"]
    if user:
        cmd.append("--user")
    cmd.append(spec)
    print(f"Installing {spec} ...")
    subprocess.check_call(cmd)
    importlib.invalidate_caches()
    importlib.import_module(pkg)
    print(f"Installed: {pkg}")


ensure("torch")
ensure("torchvision")
ensure("cv2", "opencv-python-headless")
ensure("matplotlib")
ensure("kornia")

if not LG.exists():
    print("Cloning LightGlue into third_party/...")
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/cvg/LightGlue.git", str(LG)]
    )
    sys.path.insert(0, str(LG))

import lightglue  # noqa: F401
from lightglue import LightGlue, SuperPoint  # noqa: F401

print("OK: lightglue", lightglue.__file__)

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
lg = ROOT / "third_party" / "LightGlue"
if lg.exists():
    sys.path.insert(0, str(lg))

from lightglue import LightGlue, SuperPoint
from lightglue.utils import numpy_image_to_torch, rbd

from app.georeferencer import Georeferencer
from app.localizer import Localizer
from app.map_fetcher import MapFetcher
from app.models import LocalizationRequest

CASE = ROOT / "examples" / "case_01_100m"
OUT = ROOT / "notebooks" / "artifacts" / "superpoint_case_01"
OUT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)
torch.set_grad_enabled(False)

## 1. Данные: query + ROI карты

Тот же `case_01_100m`. Карту берём из кэша/сети, ROI — как в Localizer (coarse → pad, иначе весь downscaled map, но для CPU ужимаем).

In [ ]:
meta = json.loads((CASE / "meta.json").read_text(encoding="utf-8"))
query_bgr = cv2.imread(str(CASE / "query.jpg"))
assert query_bgr is not None

req = LocalizationRequest(**meta)
h, w = query_bgr.shape[:2]
plan = Georeferencer().plan(req, (w, h))

map_raster = MapFetcher(cache_dir=ROOT / ".cache" / "tiles").fetch(plan)
loc = Localizer()

query_rot, _ = loc._rotate_image(query_bgr, -plan.yaw_deg)
map_work, _ = loc._downscale(map_raster.image, 1600)  # меньше для CPU
query_work, _ = loc._downscale(query_rot, 800)

coarse = loc._coarse_locate(query_work, map_work)
if coarse is not None and coarse[4] >= 0.25:
    cx, cy, cw, ch, score = coarse
    pad = int(0.4 * max(cw, ch))
    x0, y0 = max(0, cx - pad), max(0, cy - pad)
    x1 = min(map_work.shape[1], cx + cw + pad)
    y1 = min(map_work.shape[0], cy + ch + pad)
    roi = map_work[y0:y1, x0:x1]
    print(f"ROI from coarse score={score:.3f} size={roi.shape[1]}x{roi.shape[0]}")
else:
    # На CPU полный map тяжёлый — центральный кроп ~1200px
    side = min(1200, map_work.shape[0], map_work.shape[1])
    y0 = (map_work.shape[0] - side) // 2
    x0 = (map_work.shape[1] - side) // 2
    roi = map_work[y0 : y0 + side, x0 : x0 + side]
    print(f"coarse weak → center crop {side}x{side}")

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(cv2.cvtColor(query_work, cv2.COLOR_BGR2RGB))
ax[0].set_title(f"query {query_work.shape[1]}x{query_work.shape[0]}")
ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"map ROI {roi.shape[1]}x{roi.shape[0]}")
ax[1].axis("off")
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "query_work.jpg"), query_work)
cv2.imwrite(str(OUT / "map_roi.jpg"), roi)

## 2. Загрузка pretrained SuperPoint + LightGlue

Веса скачаются сами при первом запуске.

In [ ]:
extractor = SuperPoint(max_num_keypoints=1024).eval().to(device)
matcher = LightGlue(features="superpoint").eval().to(device)
print("SuperPoint + LightGlue ready")

## 3. SuperPoint: keypoints на query и ROI

In [ ]:
def bgr_to_torch_rgb(bgr: np.ndarray) -> torch.Tensor:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return numpy_image_to_torch(rgb)


img0 = bgr_to_torch_rgb(query_work).to(device)
img1 = bgr_to_torch_rgb(roi).to(device)

t0 = time.perf_counter()
feats0 = extractor.extract(img0)
feats1 = extractor.extract(img1)
t_extract = time.perf_counter() - t0

f0 = rbd(feats0)
f1 = rbd(feats1)
kpts0 = f0["keypoints"].detach().cpu().numpy()
kpts1 = f1["keypoints"].detach().cpu().numpy()
scores0 = f0["keypoint_scores"].detach().cpu().numpy()

print(f"extract time: {t_extract:.1f}s on {device}")
print(f"SuperPoint keypoints: query={len(kpts0)}  ROI={len(kpts1)}")
print(f"descriptor dim: {f0['descriptors'].shape[-1]}")


def draw_torch_kpts(bgr, kpts, scores=None, top_n=None, radius=3):
    vis = bgr.copy()
    order = np.arange(len(kpts))
    if scores is not None and top_n is not None:
        order = np.argsort(-scores)[:top_n]
    for i in order:
        x, y = int(round(float(kpts[i, 0]))), int(round(float(kpts[i, 1])))
        cv2.circle(vis, (x, y), radius, (0, 255, 255), 1, cv2.LINE_AA)
        cv2.circle(vis, (x, y), 1, (0, 0, 255), -1, cv2.LINE_AA)
    return vis


vis_q = draw_torch_kpts(query_work, kpts0, scores0)
vis_r = draw_torch_kpts(roi, kpts1)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(cv2.cvtColor(vis_q, cv2.COLOR_BGR2RGB))
ax[0].set_title(f"SuperPoint on query (n={len(kpts0)})")
ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(vis_r, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"SuperPoint on ROI (n={len(kpts1)})")
ax[1].axis("off")
plt.tight_layout()
plt.show()

# Zoom вокруг самой сильной точки
best = int(np.argmax(scores0))
bx, by = int(kpts0[best, 0]), int(kpts0[best, 1])
zw = 120
hq, wq = query_work.shape[:2]
zx0 = int(np.clip(bx - zw // 2, 0, max(0, wq - zw)))
zy0 = int(np.clip(by - zw // 2, 0, max(0, hq - zw)))
zx1, zy1 = zx0 + zw, zy0 + zw

overview = query_work.copy()
cv2.rectangle(overview, (zx0, zy0), (zx1, zy1), (0, 255, 0), 2)

crop = vis_q[zy0:zy1, zx0:zx1]
zoom = cv2.resize(crop, None, fx=4, fy=4, interpolation=cv2.INTER_NEAREST)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(overview, cv2.COLOR_BGR2RGB))
ax[0].set_title("query + zoom box")
ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(zoom, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"zoom ×4 around best score={scores0[best]:.3f}")
ax[1].axis("off")
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "sp_keypoints_query.jpg"), vis_q)
cv2.imwrite(str(OUT / "sp_keypoints_zoom.jpg"), zoom)

## 4. LightGlue matching (+ RANSAC для наглядности)

In [ ]:
t0 = time.perf_counter()
matches01 = matcher({"image0": feats0, "image1": feats1})
t_match = time.perf_counter() - t0
matches01 = rbd(matches01)
matches = matches01["matches"].detach().cpu().numpy()

mk0 = kpts0[matches[:, 0]]
mk1 = kpts1[matches[:, 1]]
print(f"LightGlue matches: {len(matches)}  time={t_match:.1f}s")

# RANSAC affine — как в вашем Localizer
if len(matches) >= 8:
    M, mask = cv2.estimateAffinePartial2D(
        mk0.reshape(-1, 1, 2).astype(np.float32),
        mk1.reshape(-1, 1, 2).astype(np.float32),
        method=cv2.RANSAC,
        ransacReprojThreshold=5.0,
        maxIters=4000,
        confidence=0.99,
    )
    inlier_mask = mask.ravel().astype(bool) if mask is not None else np.zeros(len(matches), bool)
    n_in = int(inlier_mask.sum())
    print(f"RANSAC inliers: {n_in}/{len(matches)}")
else:
    M, inlier_mask, n_in = None, np.zeros(len(matches), bool), 0


def draw_matches_side_by_side(img_a, img_b, pts_a, pts_b, inliers=None, max_draw=80):
    ha, wa = img_a.shape[:2]
    hb, wb = img_b.shape[:2]
    canvas = np.zeros((max(ha, hb), wa + wb, 3), dtype=np.uint8)
    canvas[:ha, :wa] = img_a
    canvas[:hb, wa : wa + wb] = img_b
    n = min(len(pts_a), max_draw)
    idx = np.linspace(0, len(pts_a) - 1, n).astype(int) if len(pts_a) else []
    for i in idx:
        p0 = (int(round(pts_a[i, 0])), int(round(pts_a[i, 1])))
        p1 = (int(round(pts_b[i, 0])) + wa, int(round(pts_b[i, 1])))
        ok = bool(inliers[i]) if inliers is not None and len(inliers) else True
        color = (0, 255, 0) if ok else (0, 0, 255)
        cv2.circle(canvas, p0, 3, color, 1, cv2.LINE_AA)
        cv2.circle(canvas, p1, 3, color, 1, cv2.LINE_AA)
        cv2.line(canvas, p0, p1, color, 1, cv2.LINE_AA)
    return canvas


match_vis = draw_matches_side_by_side(query_work, roi, mk0, mk1, inlier_mask)
plt.figure(figsize=(16, 7))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title(f"SuperPoint+LightGlue matches (green≈RANSAC inliers)  n={len(matches)} inliers={n_in}")
plt.axis("off")
plt.show()

cv2.imwrite(str(OUT / "sp_lightglue_matches.jpg"), match_vis)
print("saved →", OUT)

## 5. Рядом: SIFT + BFMatcher на тех же картинках

Чтобы глазами сравнить классику и SuperPoint.

In [ ]:
t0 = time.perf_counter()
gray0 = loc._preprocess(query_work)
gray1 = loc._preprocess(roi)
kp_s0, des_s0, _ = loc._detect_describe(gray0)
kp_s1, des_s1, _ = loc._detect_describe(gray1)

bf = cv2.BFMatcher(cv2.NORM_L2)
knn = bf.knnMatch(des_s0, des_s1, k=2)
good = []
for pair in knn:
    if len(pair) < 2:
        continue
    m, n = pair
    if m.distance < 0.75 * n.distance:
        good.append(m)

src = np.float32([kp_s0[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst = np.float32([kp_s1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
Ms, mask_s = cv2.estimateAffinePartial2D(
    src, dst, method=cv2.RANSAC, ransacReprojThreshold=5.0, maxIters=4000, confidence=0.99
)
t_sift = time.perf_counter() - t0
n_in_s = int(mask_s.ravel().sum()) if mask_s is not None else 0

sift_vis = cv2.drawMatches(
    query_work,
    kp_s0,
    roi,
    kp_s1,
    good[:80],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

fig, ax = plt.subplots(2, 1, figsize=(16, 12))
ax[0].imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
ax[0].set_title(f"SuperPoint+LightGlue  matches={len(matches)} inliers={n_in}  extract+match≈{t_extract+t_match:.1f}s")
ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(sift_vis, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"SIFT+BFMatcher     good={len(good)} inliers={n_in_s}  time≈{t_sift:.1f}s")
ax[1].axis("off")
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "sift_matches.jpg"), sift_vis)
print("Готово. Артефакты:", OUT)